In [ ]:
!apt update -q && apt-get install -qy openjdk-21-jdk-headless > /dev/null 2>&1

import subprocess
from IPython.core.magic import register_cell_magic

class PersistentJShell:
    def __init__(self):
        self.proc = subprocess.Popen(
            ["jshell", "-s"],
            stdin=subprocess.PIPE,
            stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT,
            text=True,
            bufsize=1
        )

    def execute(self, code):
        if self.proc.poll() is not None:
            raise RuntimeError("JShell process has terminated.")

        marker = "___JSHELL_CELL_END___"
        payload = code.strip() + f'\nSystem.out.println("{marker}");\n'
        self.proc.stdin.write(payload)
        self.proc.stdin.flush()

        lines = []
        while True:
            line = self.proc.stdout.readline()
            if not line:
                break
            if marker in line:
                break
            lines.append(line)

        return "".join(lines)

_jshell_instance = PersistentJShell()

@register_cell_magic
def jshell(line, cell):
    out = _jshell_instance.execute(cell)
    if out:
        print(out, end="")


In [ ]:
%%jshell

void runPy(String code) throws Exception {
    String marker = "___END_OF_PYTHON_CELL___";

    // Base64 encode code payload to bypass all quote & newline syntax errors
    String b64 = java.util.Base64.getEncoder().encodeToString(code.getBytes(java.nio.charset.StandardCharsets.UTF_8));

    // Guarantee marker printing using try...finally and display trailing expressions using ast
    String payload = String.format(
        "import base64, ast, traceback\n" +
        "try:\n" +
        "    code_str = base64.b64decode('%s').decode('utf-8')\n" +
        "    tree = ast.parse(code_str)\n" +
        "    if tree.body and isinstance(tree.body[-1], ast.Expr):\n" +
        "        exec(compile(ast.Module(body=tree.body[:-1], type_ignores=[]), '<string>', 'exec'), globals())\n" +
        "        res = eval(compile(ast.Expression(body=tree.body[-1].value), '<string>', 'eval'), globals())\n" +
        "        if res is not None:\n" +
        "            print(res)\n" +
        "    else:\n" +
        "        exec(compile(tree, '<string>', 'exec'), globals())\n" +
        "except Exception:\n" +
        "    traceback.print_exc()\n" +
        "finally:\n" +
        "    print('%s', flush=True)\n",
        b64, marker
    );

    pyIn.println(payload);

    String line;
    while ((line = pyOut.readLine()) != null) {
        if (line.contains(marker)) break;
        System.out.println(line);
    }
}

import java.io.BufferedReader;
import java.io.InputStreamReader;
import java.io.PrintStream;
import java.io.IOException;

Process pythonProcess;
PrintStream pyIn;
BufferedReader pyOut;

try {
    ProcessBuilder pb = new ProcessBuilder("python3", "-i", "-u");
    // Redirect stderr to stdout to prevent pipe deadlock and capture Python errors
    pb.redirectErrorStream(true);
    pythonProcess = pb.start();

    pyIn = new PrintStream(pythonProcess.getOutputStream(), true);
    pyOut = new BufferedReader(new InputStreamReader(pythonProcess.getInputStream()));

    System.out.println("Python interpreter initialized safely.");
} catch (IOException e) {
    e.printStackTrace();
}

In [ ]:
%%jshell

runPy("""
# Google Colab setup: fetch this repository and use this notebook's directory.
from pathlib import Path
import os

REPO_ROOT = Path('/content/BITS_programming')
if not REPO_ROOT.exists():os.system('git clone https://github.com/aqwertyuiop48/BITS_programming.git /content/BITS_programming');

NOTEBOOK_DIR = REPO_ROOT / 'assignments/assignment_13/class1/class1'
os.chdir(NOTEBOOK_DIR)
print(f'Working directory: {NOTEBOOK_DIR}')
""");


# Part 1: CNN Model Baseline - Performance & Testing

## Overview
In this notebook, we will:
1. Train a simple CNN model on an ADAS (Advanced Driver Assistance Systems) dataset
2. Log key performance metrics: accuracy, inference time, model size, parameters
3. Understand the relationship between model complexity and performance
4. Establish a baseline for robustness testing

## Key Concepts
- **Time Complexity**: How inference time grows with input size
- **Model Size vs Performance**: The tradeoff between model complexity and accuracy
- **Baseline Metrics**: Essential measurements for ML model evaluation

## Step 1: Import Libraries & Setup

In [ ]:
%%jshell

runPy("""
import os
import time
import json
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

# PyTorch imports
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision.transforms as transforms
from torch.utils.data import DataLoader
from torchvision.datasets import ImageFolder
from torchvision.models import resnet18

# Progress bar
from tqdm import tqdm

# Set random seeds for reproducibility
np.random.seed(42)
torch.manual_seed(42)

print("✅ All libraries imported successfully")
""");


## Step 2: Device Configuration

In [ ]:
%%jshell

runPy("""
# Device configuration
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

if device.type == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"CUDA Version: {torch.version.cuda}")
else:
    print("Running on CPU - training may be slower")
""");


## Step 3: Load Dataset

In [ ]:
%%jshell

runPy("""
# Dataset paths
DATASET_PATH = str(REPO_ROOT / "module_4/dataset/dataset")
TRAIN_PATH = os.path.join(DATASET_PATH, "train")
VAL_PATH = os.path.join(DATASET_PATH, "val")
TEST_PATH = os.path.join(DATASET_PATH, "test")

print(f"📁 Dataset path: {DATASET_PATH}")
""");


In [ ]:
%%jshell

runPy("""
# Data transformations
transform = transforms.Compose([
    transforms.Resize((128, 128)),  # Resize to fixed size
    transforms.ToTensor(),           # Convert to tensor (0-1 range)
    transforms.Normalize(            # Normalize with ImageNet stats
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

print("✅ Data transformations defined")
""");


In [ ]:
%%jshell

runPy("""
# Load datasets
print("Loading datasets...")
train_dataset = ImageFolder(TRAIN_PATH, transform=transform)
val_dataset = ImageFolder(VAL_PATH, transform=transform)
test_dataset = ImageFolder(TEST_PATH, transform=transform)

print(f"✅ Train samples: {len(train_dataset)}")
print(f"✅ Val samples: {len(val_dataset)}")
print(f"✅ Test samples: {len(test_dataset)}")

# Class names
class_names = train_dataset.classes
num_classes = len(class_names)
print(f"\n📚 Classes ({num_classes}): {', '.join(class_names)}")
""");


In [ ]:
%%jshell

runPy("""
# Data loaders
BATCH_SIZE = 16
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=0)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

print(f"✅ Data loaders created with batch size: {BATCH_SIZE}")
""");


## Step 4: Create Model Architecture

In [ ]:
%%jshell

runPy("""
# Create model
model = resnet18(pretrained=True)  # Use pre-trained weights
model.fc = nn.Linear(model.fc.in_features, num_classes)  # Modify final layer
model = model.to(device)

print("✅ ResNet-18 model created")
print(f"   Modified final layer to output {num_classes} classes")
""");


### Architecture Overview

We use **ResNet-18**, a pre-trained convolutional neural network:
- Input size: 128×128×3 (RGB images)
- Output: 7 classes (ADAS categories)
- Pre-trained on ImageNet for better feature extraction
- Lightweight: good balance between accuracy and speed

## Step 5: Analyze Model Complexity

In [ ]:
%%jshell

runPy("""
def count_parameters(model):
    """Count total trainable parameters"""
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

def get_model_size_mb(model):
    """Get model size in MB"""
    torch.save(model.state_dict(), "temp_model.pth")
    size_mb = os.path.getsize("temp_model.pth") / (1024 * 1024)
    os.remove("temp_model.pth")
    return size_mb

total_params = count_parameters(model)
model_size = get_model_size_mb(model)

print(f"📊 Model Statistics:")
print(f"  Total Parameters: {total_params:,}")
print(f"  Model Size: {model_size:.2f} MB")
print(f"  Average parameter size: {(model_size * 1024 * 1024 / total_params):.2f} bytes")

# Store for later
metrics = {
    'total_parameters': total_params,
    'model_size_mb': model_size
}
""");


## Step 6: Setup Training Configuration

In [ ]:
%%jshell

runPy("""
# Loss function and optimizer
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.01)

print("✅ Loss function: CrossEntropyLoss")
print("✅ Optimizer: Adam (lr=0.01)")
""");


## Step 7: Define Training & Evaluation Functions

In [ ]:
%%jshell

runPy("""
def train_epoch(model, loader, criterion, optimizer, device):
    """Train for one epoch"""
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0

    start_time = time.time()

    for images, labels in tqdm(loader, desc="Training", leave=False):
        images, labels = images.to(device), labels.to(device)

        # Forward pass
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)

        # Backward pass
        loss.backward()
        optimizer.step()

        # Statistics
        running_loss += loss.item()
        _, predicted = torch.max(outputs.data, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

    epoch_time = time.time() - start_time
    avg_loss = running_loss / len(loader)
    accuracy = 100 * correct / total

    return avg_loss, accuracy, epoch_time

print("✅ train_epoch() defined")
""");


In [ ]:
%%jshell

runPy("""
def evaluate(model, loader, criterion, device):
    """Evaluate on validation/test set"""
    model.eval()
    correct = 0
    total = 0
    running_loss = 0.0

    start_time = time.time()

    with torch.no_grad():
        for images, labels in tqdm(loader, desc="Evaluating", leave=False):
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            loss = criterion(outputs, labels)

            running_loss += loss.item()
            _, predicted = torch.max(outputs.data, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()

    eval_time = time.time() - start_time
    avg_loss = running_loss / len(loader)
    accuracy = 100 * correct / total

    return avg_loss, accuracy, eval_time

print("✅ evaluate() defined")
""");


## Step 8: Train the Model

In [ ]:
%%jshell

runPy("""
NUM_EPOCHS = 30
history = {
    'train_loss': [],
    'train_acc': [],
    'val_loss': [],
    'val_acc': [],
    'train_time': [],
    'val_time': []
}

print(f"Starting training for {NUM_EPOCHS} epochs...\n")
overall_start = time.time()

for epoch in range(NUM_EPOCHS):
    # Train
    train_loss, train_acc, train_time = train_epoch(
        model, train_loader, criterion, optimizer, device
    )

    # Validate
    val_loss, val_acc, val_time = evaluate(
        model, val_loader, criterion, device
    )

    # Store history
    history['train_loss'].append(train_loss)
    history['train_acc'].append(train_acc)
    history['val_loss'].append(val_loss)
    history['val_acc'].append(val_acc)
    history['train_time'].append(train_time)
    history['val_time'].append(val_time)

    print(f"Epoch [{epoch+1}/{NUM_EPOCHS}]")
    print(f"  Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.2f}%")
    print(f"  Val Loss:   {val_loss:.4f} | Val Acc: {val_acc:.2f}%")
    print(f"  Time: Train {train_time:.2f}s | Val {val_time:.2f}s\n")

overall_time = time.time() - overall_start
print(f"✅ Training completed in {overall_time:.2f} seconds")

metrics['total_training_time'] = overall_time
""");


## Step 9: Test Evaluation

In [ ]:
%%jshell

runPy("""
# Test evaluation
test_loss, test_acc, test_time = evaluate(model, test_loader, criterion, device)

# Calculate per-image inference time
num_test_samples = len(test_dataset)
avg_inference_time = (test_time / num_test_samples) * 1000  # in milliseconds

print("\n" + "="*60)
print("📊 TEST SET PERFORMANCE")
print("="*60)
print(f"Test Loss: {test_loss:.4f}")
print(f"Test Accuracy: {test_acc:.2f}%")
print(f"Total Inference Time: {test_time:.2f} seconds")
print(f"Avg Inference Time per Image: {avg_inference_time:.4f} ms")
print("="*60 + "\n")

# Update metrics
metrics['test_accuracy'] = test_acc
metrics['total_inference_time'] = test_time
metrics['avg_inference_time_ms'] = avg_inference_time
""");


## Step 10: Visualize Training History

In [ ]:
%%jshell

runPy("""
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Loss curves
axes[0].plot(history['train_loss'], label='Train Loss', marker='o', linewidth=2)
axes[0].plot(history['val_loss'], label='Val Loss', marker='s', linewidth=2)
axes[0].set_xlabel('Epoch', fontsize=12)
axes[0].set_ylabel('Loss', fontsize=12)
axes[0].set_title('Training & Validation Loss', fontsize=14, fontweight='bold')
axes[0].legend(fontsize=11)
axes[0].grid(True, alpha=0.3)

# Accuracy curves
axes[1].plot(history['train_acc'], label='Train Accuracy', marker='o', linewidth=2)
axes[1].plot(history['val_acc'], label='Val Accuracy', marker='s', linewidth=2)
axes[1].set_xlabel('Epoch', fontsize=12)
axes[1].set_ylabel('Accuracy (%)', fontsize=12)
axes[1].set_title('Training & Validation Accuracy', fontsize=14, fontweight='bold')
axes[1].legend(fontsize=11)
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('training_curves.png', dpi=150, bbox_inches='tight')
plt.savefig('/content/regression_plot.png')

print("✅ Training curves saved as 'training_curves.png'")
""");


## Step 11: Final Summary Report

In [ ]:
%%jshell

runPy("""
print("\n" + "="*70)
print("🎯 BASELINE MODEL PERFORMANCE REPORT")
print("="*70)

print("\n📈 Accuracy Metrics:")
print(f"  Train Accuracy (final epoch): {history['train_acc'][-1]:.2f}%")
print(f"  Val Accuracy (final epoch):   {history['val_acc'][-1]:.2f}%")
print(f"  Test Accuracy:                {metrics['test_accuracy']:.2f}%")

print("\n⏱️  Time Metrics:")
print(f"  Total Training Time:          {metrics['total_training_time']:.2f} seconds")
print(f"  Total Inference Time:         {metrics['total_inference_time']:.2f} seconds")
print(f"  Avg Inference per Image:      {metrics['avg_inference_time_ms']:.4f} ms")

print("\n📊 Model Complexity:")
print(f"  Total Parameters:             {metrics['total_parameters']:,}")
print(f"  Model Size:                   {metrics['model_size_mb']:.2f} MB")

# Analysis
train_val_gap = history['train_acc'][-1] - history['val_acc'][-1]

print("\n🔑 Key Observations:")
print(f"  Train-Val Accuracy Gap:       {train_val_gap:.2f}%")

if train_val_gap > 10:
    print("  ⚠️  High overfitting detected (Train >> Val accuracy)")
elif train_val_gap > 5:
    print("  ⚠️  Some overfitting detected")
else:
    print("  ✅ Good generalization (Train and Val accuracy are close)")

print("\n💡 This baseline will be used for:")
print("  • Overfitting detection in Part 2")
print("  • Robustness testing in Part 3")
print("  • Comparing with regularization techniques")

print("\n" + "="*70)
""");


In [ ]:
%%jshell

runPy("""
import datetime, pytz;
print("Current Time in IST:", datetime.datetime.now(pytz.utc).astimezone(pytz.timezone('Asia/Kolkata')).strftime('%Y-%m-%d %H:%M:%S'))
""");
